# 01. Load the Olist data into MySQL

**Purpose:** Load the 9 Olist CSV files into a MySQL database called `olist` with explicit data types, reconcile row counts against the CSVs, and add primary keys and indexes.

**Inputs:** the 9 CSVs in `data/raw/` (not committed; download from Kaggle) and database credentials in `.env`.

**Outputs:** 9 tables in the MySQL database `olist`.

**Run order:** first notebook. Create the database with `sql/00_setup.sql` before running it.

## 1. Connect to MySQL

Credentials are read from `.env`, so no password appears in the code.

In [1]:
import os
import time
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.types import VARCHAR, DATETIME, INTEGER, NUMERIC

# Notebooks run from notebooks/, so the project root is one level up
ROOT = Path.cwd().parent
load_dotenv(ROOT / ".env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"
)

RAW_DIR = ROOT / "data" / "raw"

assert RAW_DIR.exists(), "Put the 9 Olist CSV files in data/raw/"


## 2. Describe each table

Types are set explicitly instead of letting pandas guess: ids are `VARCHAR(32)` so they can be indexed (a `TEXT` column cannot), money is `NUMERIC` to avoid rounding errors, timestamps become real `DATETIME` values, and zip prefixes stay text so leading zeros survive.

In [2]:
ID = VARCHAR(32)    # all Olist ids are 32-character hex strings
ZIP = VARCHAR(5)    # zip prefixes are 5 digits, kept as text so leading zeros survive

TABLES = {
    "orders": {
        "file": "olist_orders_dataset.csv",
        "dates": ["order_purchase_timestamp", "order_approved_at",
                  "order_delivered_carrier_date", "order_delivered_customer_date",
                  "order_estimated_delivery_date"],
        "read_dtype": None,
        "sql_dtype": {
            "order_id": ID, "customer_id": ID, "order_status": VARCHAR(20),
            "order_purchase_timestamp": DATETIME(), "order_approved_at": DATETIME(),
            "order_delivered_carrier_date": DATETIME(),
            "order_delivered_customer_date": DATETIME(),
            "order_estimated_delivery_date": DATETIME(),
        },
    },
    "order_items": {
        "file": "olist_order_items_dataset.csv",
        "dates": ["shipping_limit_date"],
        "read_dtype": None,
        "sql_dtype": {
            "order_id": ID, "order_item_id": INTEGER(), "product_id": ID,
            "seller_id": ID, "shipping_limit_date": DATETIME(),
            "price": NUMERIC(10, 2), "freight_value": NUMERIC(10, 2),
        },
    },
    "order_payments": {
        "file": "olist_order_payments_dataset.csv",
        "dates": [],
        "read_dtype": None,
        "sql_dtype": {
            "order_id": ID, "payment_sequential": INTEGER(),
            "payment_type": VARCHAR(20), "payment_installments": INTEGER(),
            "payment_value": NUMERIC(10, 2),
        },
    },
    "order_reviews": {
        "file": "olist_order_reviews_dataset.csv",
        "dates": ["review_creation_date", "review_answer_timestamp"],
        "read_dtype": None,
        "sql_dtype": {
            "review_id": ID, "order_id": ID, "review_score": INTEGER(),
            "review_creation_date": DATETIME(), "review_answer_timestamp": DATETIME(),
        },
    },
    "customers": {
        "file": "olist_customers_dataset.csv",
        "dates": [],
        "read_dtype": {"customer_zip_code_prefix": str},
        "sql_dtype": {
            "customer_id": ID, "customer_unique_id": ID,
            "customer_zip_code_prefix": ZIP, "customer_city": VARCHAR(100),
            "customer_state": VARCHAR(2),
        },
    },
    "sellers": {
        "file": "olist_sellers_dataset.csv",
        "dates": [],
        "read_dtype": {"seller_zip_code_prefix": str},
        "sql_dtype": {
            "seller_id": ID, "seller_zip_code_prefix": ZIP,
            "seller_city": VARCHAR(100), "seller_state": VARCHAR(2),
        },
    },
    "products": {
        "file": "olist_products_dataset.csv",
        "dates": [],
        "read_dtype": None,
        "sql_dtype": {"product_id": ID, "product_category_name": VARCHAR(100)},
    },
    "geolocation": {
        "file": "olist_geolocation_dataset.csv",
        "dates": [],
        "read_dtype": {"geolocation_zip_code_prefix": str},
        "sql_dtype": {
            "geolocation_zip_code_prefix": ZIP,
            "geolocation_city": VARCHAR(100), "geolocation_state": VARCHAR(2),
        },
    },
    "category_translation": {
        "file": "product_category_name_translation.csv",
        "dates": [],
        "read_dtype": None,
        "sql_dtype": {
            "product_category_name": VARCHAR(100),
            "product_category_name_english": VARCHAR(100),
        },
    },
}

## 3. Load all tables and reconcile row counts

Each CSV is loaded in batches of 10,000 rows, then the row count in MySQL is compared with the CSV. `if_exists="replace"` makes the notebook safe to re-run.

In [3]:
for table, cfg in TABLES.items():
    start = time.time()
    df = pd.read_csv(RAW_DIR / cfg["file"],
                     parse_dates=cfg["dates"],
                     dtype=cfg["read_dtype"])
    df.to_sql(table, engine, if_exists="replace", index=False,
              dtype=cfg["sql_dtype"], chunksize=10000)

    with engine.connect() as conn:
        n_db = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
    status = "OK" if n_db == len(df) else "MISMATCH - INVESTIGATE"
    print(f"{table:22s} csv_rows={len(df):>9,}  db_rows={n_db:>9,}  {status}  ({time.time()-start:.1f}s)")

orders                 csv_rows=   99,441  db_rows=   99,441  OK  (9.0s)
order_items            csv_rows=  112,650  db_rows=  112,650  OK  (7.0s)
order_payments         csv_rows=  103,886  db_rows=  103,886  OK  (5.0s)
order_reviews          csv_rows=   99,224  db_rows=   99,224  OK  (7.2s)
customers              csv_rows=   99,441  db_rows=   99,441  OK  (4.6s)
sellers                csv_rows=    3,095  db_rows=    3,095  OK  (0.2s)
products               csv_rows=   32,951  db_rows=   32,951  OK  (2.4s)
geolocation            csv_rows=1,000,163  db_rows=1,000,163  OK  (48.5s)
category_translation   csv_rows=       71  db_rows=       71  OK  (0.1s)


## 4. Primary keys and indexes

Primary keys enforce uniqueness (`orders.order_id`, and the pair `order_id` + `order_item_id` for items). Indexes on the join columns make the joins in `sql/` fast. `order_reviews` gets a plain index only, because `review_id` repeats in this dataset.

In [4]:
INDEXES = [
    "ALTER TABLE orders ADD PRIMARY KEY (order_id)",
    "CREATE INDEX idx_orders_customer ON orders (customer_id)",
    "CREATE INDEX idx_orders_status ON orders (order_status)",
    "CREATE INDEX idx_orders_purchase ON orders (order_purchase_timestamp)",
    "ALTER TABLE order_items ADD PRIMARY KEY (order_id, order_item_id)",
    "CREATE INDEX idx_items_seller ON order_items (seller_id)",
    "CREATE INDEX idx_items_product ON order_items (product_id)",
    "CREATE INDEX idx_payments_order ON order_payments (order_id)",
    "CREATE INDEX idx_reviews_order ON order_reviews (order_id)",
    "ALTER TABLE customers ADD PRIMARY KEY (customer_id)",
    "CREATE INDEX idx_customers_unique ON customers (customer_unique_id)",
    "ALTER TABLE sellers ADD PRIMARY KEY (seller_id)",
    "ALTER TABLE products ADD PRIMARY KEY (product_id)",
    "CREATE INDEX idx_geo_zip ON geolocation (geolocation_zip_code_prefix)",
    "CREATE INDEX idx_trans_cat ON category_translation (product_category_name)",
]

for stmt in INDEXES:
    try:
        with engine.begin() as conn:
            conn.execute(text(stmt))
        print("done:", stmt)
    except Exception as e:
        print("FAILED:", stmt, "\n   reason:", str(e)[:150])

done: ALTER TABLE orders ADD PRIMARY KEY (order_id)
done: CREATE INDEX idx_orders_customer ON orders (customer_id)
done: CREATE INDEX idx_orders_status ON orders (order_status)
done: CREATE INDEX idx_orders_purchase ON orders (order_purchase_timestamp)
done: ALTER TABLE order_items ADD PRIMARY KEY (order_id, order_item_id)
done: CREATE INDEX idx_items_seller ON order_items (seller_id)
done: CREATE INDEX idx_items_product ON order_items (product_id)
done: CREATE INDEX idx_payments_order ON order_payments (order_id)
done: CREATE INDEX idx_reviews_order ON order_reviews (order_id)
done: ALTER TABLE customers ADD PRIMARY KEY (customer_id)
done: CREATE INDEX idx_customers_unique ON customers (customer_unique_id)
done: ALTER TABLE sellers ADD PRIMARY KEY (seller_id)
done: ALTER TABLE products ADD PRIMARY KEY (product_id)
done: CREATE INDEX idx_geo_zip ON geolocation (geolocation_zip_code_prefix)
done: CREATE INDEX idx_trans_cat ON category_translation (product_category_name)


## Result

All 9 tables loaded and every row count matches its CSV (orders 99,441; order_items 112,650; order_payments 103,886; order_reviews 99,224; customers 99,441; sellers 3,095; products 32,951; geolocation 1,000,163; category_translation 71). Primary keys and indexes are in place.